# COMET vs STOmics Cellbin Segmentation Comparison

Compare COMET cell segmentation (Visiopharm, warped via VALIS) against STOmics DAPI-derived cell segmentation (cellbin GEF) in the same coordinate space.

**Pilot sample:** SO34 (A03979E2) - 552K COMET cells, 504K STOmics cells

**Goals:**
1. Validate prerequisites (cellbin exists, alignment exists)
2. Export QuPath-compatible GeoJSON for visual alignment inspection
3. Rasterize both masks and compute spatial overlap
4. Match cells between segmentations (1:1, fragmented, merged)
5. Compare cell morphology and expression concordance

## 1. Setup & Configuration

In [ ]:
import sys
import json
import logging
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

sys.path.insert(0, '..')
from alignment_utils import (
    validate_cellbin_comparison_inputs,
    load_stomics_cellbin_borders,
    rasterize_cellbin_to_mask,
    export_cellbin_borders_to_geojson,
    rasterize_geojson_to_mask,
    load_geojson,
    geojson_centroids,
    compare_segmentation_cells,
    compute_mask_overlap_metrics,
    compare_matched_cell_expression,
    load_stomics_cellbin_gef,
    aggregate_transcripts_by_mask,
    DefaultPaths,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
# ============= CONFIGURATION =============
SAMPLE_ID = 'SO34'
CHIP_ID = 'A03979E2'

# Paths (use /mnt/t/ for WSL2)
BASE = Path('/mnt/t/Sammy Data')
CELLBIN_PATH = BASE / CHIP_ID / '03.ssDNA_analysis' / f'{CHIP_ID}.adjusted.cellbin.gef'
DAPI_PATH = BASE / CHIP_ID / '03.ssDNA_analysis' / f'ssDNA_{CHIP_ID}_regist.tif'
TISSUE_GEF_PATH = BASE / CHIP_ID / '01.StandardWorkflow_Result' / 'GeneExpMatrix' / f'{CHIP_ID}.tissue.gef'

OUTPUT_DIR = BASE / 'projects' / 'out' / 'comet_stomics_alignment' / f'{SAMPLE_ID}_{CHIP_ID}'
WARPED_GEOJSON_PATH = OUTPUT_DIR / f'{SAMPLE_ID}_warped_segmentations.geojson'

COMPARISON_DIR = OUTPUT_DIR / 'cellbin_comparison'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

# STOmics DAPI dimensions (for mask rasterization)
DAPI_SHAPE = (23520, 23520)

# Matching parameters
MAX_MATCH_DISTANCE = 30  # pixels

print(f"Sample: {SAMPLE_ID} ({CHIP_ID})")
print(f"Cellbin: {CELLBIN_PATH}")
print(f"Warped GeoJSON: {WARPED_GEOJSON_PATH}")
print(f"Output: {COMPARISON_DIR}")

## 2. Preflight Validation

Verify all required data is present and usable before proceeding.

In [ ]:
status = validate_cellbin_comparison_inputs(
    SAMPLE_ID, CHIP_ID, OUTPUT_DIR,
    cellbin_path=CELLBIN_PATH, dapi_path=DAPI_PATH
)

# Display as formatted table
print(f"\n{'Check':<20} {'Status':<6} {'Details'}")
print("-" * 80)
for name, check in status['checks'].items():
    symbol = 'PASS' if check['pass'] else 'FAIL'
    print(f"{name:<20} [{symbol}]  {check['message']}")
print("-" * 80)
print(f"Overall: {'READY - proceeding' if status['ready'] else 'NOT READY - fix issues above'}")

assert status['ready'], "Preflight validation failed — cannot proceed"

## 3. QuPath Alignment Export

Export STOmics cellbin borders as GeoJSON for visual inspection in QuPath alongside the COMET warped GeoJSON.

In [ ]:
# Export subsampled STOmics cellbin GeoJSON for QuPath
stomics_geojson_path = COMPARISON_DIR / f'{SAMPLE_ID}_stomics_cellbin_borders.geojson'

# subsample=10 exports every 10th cell (~50K features) for faster QuPath loading
t0 = time.time()
stomics_geojson = export_cellbin_borders_to_geojson(
    CELLBIN_PATH, stomics_geojson_path, subsample=10
)
print(f"Export time: {time.time()-t0:.1f}s")

print(f"\n=== QuPath Visual Inspection ===")
print(f"Load these files in QuPath on the STOmics DAPI image:")
print(f"  1. COMET warped:   {WARPED_GEOJSON_PATH}")
print(f"  2. STOmics cellbin: {stomics_geojson_path}")
print(f"\nUse different colors to compare alignment quality.")

## 4. Load COMET Data

Load the VALIS-warped COMET GeoJSON, extract centroids, and rasterize to mask.

In [ ]:
# Load COMET warped GeoJSON
t0 = time.time()
comet_geojson = load_geojson(WARPED_GEOJSON_PATH)
comet_centroids, comet_labels = geojson_centroids(comet_geojson)
t1 = time.time()
print(f"COMET: {len(comet_centroids):,} cells loaded ({t1-t0:.1f}s)")

# Rasterize COMET mask
t2 = time.time()
comet_mask, comet_pheno_map = rasterize_geojson_to_mask(comet_geojson, DAPI_SHAPE)
t3 = time.time()
comet_n_unique = len(np.unique(comet_mask)) - 1
comet_fg_pct = (comet_mask > 0).sum() / comet_mask.size * 100
print(f"COMET mask: {comet_n_unique:,} unique labels, {comet_fg_pct:.1f}% foreground ({t3-t2:.1f}s)")

## 5. Load STOmics Cellbin

Load border polygons, extract centroids, and rasterize to mask.

In [ ]:
# Load STOmics cellbin borders
t0 = time.time()
cellbin_data = load_stomics_cellbin_borders(CELLBIN_PATH)
stomics_centroids = cellbin_data['centroids']
t1 = time.time()
print(f"STOmics: {cellbin_data['n_cells']:,} cells loaded ({t1-t0:.1f}s)")

# Rasterize STOmics mask
t2 = time.time()
stomics_mask = rasterize_cellbin_to_mask(cellbin_data, DAPI_SHAPE)
t3 = time.time()
stomics_n_unique = len(np.unique(stomics_mask)) - 1
stomics_fg_pct = (stomics_mask > 0).sum() / stomics_mask.size * 100
print(f"STOmics mask: {stomics_n_unique:,} unique labels, {stomics_fg_pct:.1f}% foreground ({t3-t2:.1f}s)")

## 6. Visual Alignment Overlay

Overlay both segmentation mask contours on the STOmics DAPI image.
- **Cyan**: COMET cell boundaries
- **Magenta**: STOmics cell boundaries
- Full tissue view + 3 zoomed ROIs for detailed inspection

In [ ]:
# Load DAPI for background
import tifffile
dapi = tifffile.imread(str(DAPI_PATH))
print(f"DAPI shape: {dapi.shape}, dtype: {dapi.dtype}")

# Create binary edge maps from masks (downsample for full-tissue view)
def mask_edges(mask, downsample=1):
    """Extract cell boundary pixels from a labeled mask."""
    from scipy.ndimage import binary_dilation
    if downsample > 1:
        mask = mask[::downsample, ::downsample]
    fg = mask > 0
    # Edge = foreground pixel adjacent to different label or background
    shifted_r = np.roll(mask, 1, axis=0)
    shifted_c = np.roll(mask, 1, axis=1)
    edges = fg & ((mask != shifted_r) | (mask != shifted_c))
    return edges

# Full tissue overlay (downsampled 4x for speed)
ds = 4
dapi_ds = dapi[::ds, ::ds]
comet_edges = mask_edges(comet_mask, ds)
stomics_edges = mask_edges(stomics_mask, ds)

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Left: full tissue with both overlays
ax = axes[0]
# Normalize DAPI for display
dapi_norm = dapi_ds.astype(np.float32)
dapi_norm = np.clip(dapi_norm / np.percentile(dapi_norm[dapi_norm > 0], 99), 0, 1)
ax.imshow(dapi_norm, cmap='gray', aspect='equal')
# Overlay edges
overlay = np.zeros((*dapi_ds.shape, 4), dtype=np.float32)
overlay[comet_edges, :] = [0, 1, 1, 0.7]    # cyan for COMET
overlay[stomics_edges, :] = [1, 0, 1, 0.7]   # magenta for STOmics
ax.imshow(overlay, aspect='equal')
ax.set_title(f'Full Tissue: COMET (cyan) vs STOmics (magenta)\n{SAMPLE_ID}', fontsize=12)
ax.legend(handles=[
    mpatches.Patch(color='cyan', label=f'COMET ({len(comet_centroids):,} cells)'),
    mpatches.Patch(color='magenta', label=f'STOmics ({cellbin_data["n_cells"]:,} cells)'),
], loc='upper right', fontsize=9)

# Right: foreground coverage comparison
ax = axes[1]
# Create 3-class map: 0=background, 1=COMET-only, 2=STOmics-only, 3=both
comet_fg = (comet_mask > 0)[::ds, ::ds]
stomics_fg = (stomics_mask > 0)[::ds, ::ds]
coverage = np.zeros_like(comet_fg, dtype=np.uint8)
coverage[comet_fg & ~stomics_fg] = 1
coverage[~comet_fg & stomics_fg] = 2
coverage[comet_fg & stomics_fg] = 3
cmap = ListedColormap(['black', 'cyan', 'magenta', 'white'])
ax.imshow(coverage, cmap=cmap, aspect='equal', vmin=0, vmax=3)
ax.set_title('Coverage Map', fontsize=12)
ax.legend(handles=[
    mpatches.Patch(color='black', label='Background'),
    mpatches.Patch(color='cyan', label='COMET only'),
    mpatches.Patch(color='magenta', label='STOmics only'),
    mpatches.Patch(color='white', label='Both'),
], loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig(COMPARISON_DIR / 'full_tissue_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Zoomed ROIs for detailed alignment inspection
# Define 3 ROI regions (y_start, y_end, x_start, x_end) in full-resolution coords
ZOOM_ROIS = [
    (8000, 9000, 10000, 11000, "Center"),
    (5000, 6000, 6000, 7000, "Upper-left tissue"),
    (15000, 16000, 14000, 15000, "Lower-right tissue"),
]

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

for idx, (y0, y1, x0, x1, label) in enumerate(ZOOM_ROIS):
    ax = axes[idx]
    
    # Crop DAPI and masks
    dapi_crop = dapi[y0:y1, x0:x1].astype(np.float32)
    dapi_crop = np.clip(dapi_crop / np.percentile(dapi_crop[dapi_crop > 0], 99) if dapi_crop.max() > 0 else dapi_crop, 0, 1)
    comet_crop = comet_mask[y0:y1, x0:x1]
    stomics_crop = stomics_mask[y0:y1, x0:x1]
    
    # Edge detection on crops
    c_edges = mask_edges(comet_crop)
    s_edges = mask_edges(stomics_crop)
    
    ax.imshow(dapi_crop, cmap='gray', aspect='equal')
    overlay = np.zeros((*dapi_crop.shape, 4), dtype=np.float32)
    overlay[c_edges, :] = [0, 1, 1, 0.9]
    overlay[s_edges, :] = [1, 0, 1, 0.9]
    ax.imshow(overlay, aspect='equal')
    
    n_comet_roi = len(np.unique(comet_crop)) - 1
    n_stomics_roi = len(np.unique(stomics_crop)) - 1
    ax.set_title(f'{label}\n[{x0}:{x1}, {y0}:{y1}]\n'
                 f'COMET: {n_comet_roi} | STOmics: {n_stomics_roi}', fontsize=10)

plt.suptitle(f'Zoomed ROIs: COMET (cyan) vs STOmics (magenta) - {SAMPLE_ID}', fontsize=13)
plt.tight_layout()
plt.savefig(COMPARISON_DIR / 'zoomed_roi_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Cell Matching

Match cells between the two segmentations using bidirectional KD-tree queries.

In [ ]:
t0 = time.time()
match_result = compare_segmentation_cells(
    comet_centroids, stomics_centroids, max_distance=MAX_MATCH_DISTANCE
)
t1 = time.time()
print(f"Matching completed in {t1-t0:.1f}s\n")

# Display summary table
summary = match_result['summary']
print(f"{'Category':<25} {'Count':>10} {'Percent':>10}")
print("-" * 50)
print(f"{'COMET cells total':<25} {summary['n_comet']:>10,}")
print(f"{'STOmics cells total':<25} {summary['n_stomics']:>10,}")
print(f"{'1:1 mutual matches':<25} {summary['n_1to1']:>10,}")
print(f"{'COMET merged':<25} {summary['n_comet_merged']:>10,}")
print(f"{'COMET only (unmatched)':<25} {summary['n_comet_only']:>10,}")
print(f"{'STOmics fragmented':<25} {summary['n_stomics_fragmented']:>10,}")
print(f"{'STOmics only (unmatched)':<25} {summary['n_stomics_only']:>10,}")
print("-" * 50)
print(f"{'COMET match rate':<25} {'':>10} {summary['pct_comet_matched']:>9.1f}%")
print(f"{'STOmics match rate':<25} {'':>10} {summary['pct_stomics_matched']:>9.1f}%")

matches_1to1 = match_result['matches_1to1']
if len(matches_1to1) > 0:
    print(f"\n1:1 match distances: median={matches_1to1['distance'].median():.1f}px, "
          f"mean={matches_1to1['distance'].mean():.1f}px, "
          f"max={matches_1to1['distance'].max():.1f}px")

## 8. Pixel Overlap Metrics

In [ ]:
overlap_metrics = compute_mask_overlap_metrics(comet_mask, stomics_mask, matches_1to1)

print(f"\n{'Metric':<30} {'Value':>15}")
print("-" * 50)
for k, v in overlap_metrics.items():
    if isinstance(v, float):
        print(f"{k:<30} {v:>15.4f}")
    else:
        print(f"{k:<30} {v:>15,}")

## 9. Cell Morphology Comparison

Compare cell sizes and match distances between the two segmentations.

In [ ]:
# Extract COMET cell areas from GeoJSON
from skimage.measure import regionprops
comet_areas = []
for feat in comet_geojson['features']:
    area = feat['properties'].get('area_px', 0)
    if area > 0:
        comet_areas.append(area)
comet_areas = np.array(comet_areas) if comet_areas else np.zeros(0)

stomics_areas = cellbin_data['areas']

fig, axes = plt.subplots(1, 3, figsize=(21, 5))

# 1. Cell size distributions
ax = axes[0]
bins = np.linspace(0, 2000, 100)
if len(comet_areas) > 0:
    ax.hist(comet_areas, bins=bins, alpha=0.6, color='cyan', label=f'COMET (n={len(comet_areas):,})', density=True)
ax.hist(stomics_areas, bins=bins, alpha=0.6, color='magenta', label=f'STOmics (n={len(stomics_areas):,})', density=True)
ax.set_xlabel('Cell Area (pixels)')
ax.set_ylabel('Density')
ax.set_title('Cell Size Distribution')
ax.legend(fontsize=9)
ax.set_xlim(0, 2000)

# 2. Match distance histogram
ax = axes[1]
if len(matches_1to1) > 0:
    ax.hist(matches_1to1['distance'], bins=50, color='steelblue', edgecolor='white')
    ax.axvline(matches_1to1['distance'].median(), color='red', ls='--',
               label=f"Median: {matches_1to1['distance'].median():.1f}px")
    ax.set_xlabel('Centroid Distance (pixels)')
    ax.set_ylabel('Count')
    ax.set_title(f'1:1 Match Distances (n={len(matches_1to1):,})')
    ax.legend()

# 3. Matched cell area scatter (COMET vs STOmics for 1:1 pairs)
ax = axes[2]
if len(matches_1to1) > 0 and len(comet_areas) > 0:
    # Get areas for matched pairs
    sample_n = min(5000, len(matches_1to1))
    sampled = matches_1to1.sample(n=sample_n, random_state=42)
    
    matched_stomics_areas = stomics_areas[sampled['stomics_idx'].values]
    
    # For COMET areas, we need to use regionprops on the mask or feature areas
    # Use the GeoJSON area if available
    matched_comet_areas = []
    for ci in sampled['comet_idx'].values:
        if ci < len(comet_geojson['features']):
            a = comet_geojson['features'][ci]['properties'].get('area_px', 0)
            matched_comet_areas.append(a)
        else:
            matched_comet_areas.append(0)
    matched_comet_areas = np.array(matched_comet_areas)
    
    valid = (matched_comet_areas > 0) & (matched_stomics_areas > 0)
    if valid.sum() > 0:
        ax.scatter(matched_comet_areas[valid], matched_stomics_areas[valid],
                   s=1, alpha=0.3, c='steelblue')
        max_area = max(matched_comet_areas[valid].max(), matched_stomics_areas[valid].max())
        ax.plot([0, max_area], [0, max_area], 'r--', alpha=0.5, label='y=x')
        ax.set_xlabel('COMET Cell Area (px)')
        ax.set_ylabel('STOmics Cell Area (px)')
        ax.set_title(f'Matched Cell Areas (n={valid.sum():,})')
        ax.set_xlim(0, min(max_area, 3000))
        ax.set_ylim(0, min(max_area, 3000))
        ax.legend()

plt.tight_layout()
plt.savefig(COMPARISON_DIR / 'morphology_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print stats
print(f"\nCell area statistics:")
if len(comet_areas) > 0:
    print(f"  COMET:   median={np.median(comet_areas):.0f}, mean={np.mean(comet_areas):.0f}, "
          f"std={np.std(comet_areas):.0f}")
print(f"  STOmics: median={np.median(stomics_areas):.0f}, mean={np.mean(stomics_areas):.0f}, "
      f"std={np.std(stomics_areas):.0f}")

## 10. Expression Concordance

Compare gene expression profiles between matched COMET and STOmics cells.

**Note:** Both AnnDatas derive from the same underlying transcript data (tissue GEF). COMET AnnData uses COMET mask to assign transcripts; STOmics AnnData uses STOmics mask. Correlations reflect mask spatial agreement.

In [ ]:
# Load STOmics cellbin expression (native STOmics segmentation)
t0 = time.time()
stomics_adata = load_stomics_cellbin_gef(CELLBIN_PATH)
t1 = time.time()
print(f"STOmics AnnData: {stomics_adata.shape} ({t1-t0:.1f}s)")

# Load or generate COMET AnnData (COMET mask-based transcript assignment)
comet_adata_path = OUTPUT_DIR / f'{SAMPLE_ID}_comet_cellbin.h5ad'
if comet_adata_path.exists():
    import anndata as ad
    comet_adata = ad.read_h5ad(str(comet_adata_path))
    print(f"COMET AnnData loaded from cache: {comet_adata.shape}")
else:
    print("Generating COMET AnnData via transcript assignment (this may take a few minutes)...")
    t2 = time.time()
    comet_adata, _ = aggregate_transcripts_by_mask(TISSUE_GEF_PATH, comet_mask)
    t3 = time.time()
    print(f"COMET AnnData: {comet_adata.shape} ({t3-t2:.1f}s)")
    # Cache for reuse
    comet_adata.write_h5ad(str(comet_adata_path))
    print(f"Cached to {comet_adata_path}")

In [ ]:
# Run expression concordance analysis
t0 = time.time()
expr_result = compare_matched_cell_expression(comet_adata, stomics_adata, matches_1to1)
t1 = time.time()
print(f"Expression comparison: {t1-t0:.1f}s\n")

# Display summary
print(f"{'Metric':<35} {'Value':>12}")
print("-" * 50)
for k, v in expr_result['summary'].items():
    if isinstance(v, float):
        print(f"{k:<35} {v:>12.3f}")
    else:
        print(f"{k:<35} {v:>12,}")

In [ ]:
# Expression concordance visualizations
per_cell = expr_result['per_cell']
per_gene = expr_result['per_gene']

fig, axes = plt.subplots(1, 3, figsize=(21, 5))

# 1. Per-cell Pearson r distribution
ax = axes[0]
if len(per_cell) > 0:
    ax.hist(per_cell['pearson_r'], bins=50, color='steelblue', edgecolor='white')
    ax.axvline(per_cell['pearson_r'].median(), color='red', ls='--',
               label=f"Median r={per_cell['pearson_r'].median():.3f}")
    ax.set_xlabel('Pearson r')
    ax.set_ylabel('Count')
    ax.set_title('Per-Cell Expression Correlation')
    ax.legend()

# 2. Gene count scatter (COMET vs STOmics per matched cell)
ax = axes[1]
if len(per_cell) > 0:
    ax.scatter(per_cell['comet_total_counts'], per_cell['stomics_total_counts'],
               s=1, alpha=0.3, c='steelblue')
    max_c = max(per_cell['comet_total_counts'].max(), per_cell['stomics_total_counts'].max())
    ax.plot([0, max_c], [0, max_c], 'r--', alpha=0.5, label='y=x')
    ax.set_xlabel('COMET Total Counts')
    ax.set_ylabel('STOmics Total Counts')
    ax.set_title('Transcript Counts per Matched Cell')
    ax.legend()

# 3. Top genes by correlation
ax = axes[2]
if len(per_gene) > 0:
    top_genes = per_gene.nlargest(20, 'pearson_r')
    ax.barh(range(len(top_genes)), top_genes['pearson_r'].values, color='steelblue')
    ax.set_yticks(range(len(top_genes)))
    ax.set_yticklabels(top_genes['gene'].values, fontsize=8)
    ax.set_xlabel('Pearson r')
    ax.set_title('Top 20 Genes by Correlation')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(COMPARISON_DIR / 'expression_concordance.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Spatial Match Type Map

Color-coded spatial map showing match categories across the tissue.

In [ ]:
# Spatial map of match types
# Plot COMET and STOmics centroids colored by match classification
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Subsample for plotting performance
n_plot = 50000

# COMET cells by classification
ax = axes[0]
c2s = match_result['comet_to_stomics']
colors_map = {'1to1': 'green', 'merged': 'orange', 'comet_only': 'red'}
for cat, color in colors_map.items():
    mask = c2s['classification'] == cat
    subset = c2s[mask]
    if len(subset) > n_plot:
        subset = subset.sample(n=n_plot, random_state=42)
    coords = comet_centroids[subset['comet_idx'].values]
    ax.scatter(coords[:, 0], coords[:, 1], s=0.1, alpha=0.3, c=color, label=f'{cat} ({mask.sum():,})')
ax.set_title(f'COMET Cells by Match Type', fontsize=12)
ax.legend(markerscale=20, fontsize=9)
ax.set_aspect('equal')
ax.invert_yaxis()
ax.set_xlabel('X (px)')
ax.set_ylabel('Y (px)')

# STOmics cells by classification
ax = axes[1]
s2c = match_result['stomics_to_comet']
colors_map_s = {'1to1': 'green', 'fragmented': 'orange', 'stomics_only': 'blue'}
for cat, color in colors_map_s.items():
    mask = s2c['classification'] == cat
    subset = s2c[mask]
    if len(subset) > n_plot:
        subset = subset.sample(n=n_plot, random_state=42)
    coords = stomics_centroids[subset['stomics_idx'].values]
    ax.scatter(coords[:, 0], coords[:, 1], s=0.1, alpha=0.3, c=color, label=f'{cat} ({mask.sum():,})')
ax.set_title(f'STOmics Cells by Match Type', fontsize=12)
ax.legend(markerscale=20, fontsize=9)
ax.set_aspect('equal')
ax.invert_yaxis()
ax.set_xlabel('X (px)')
ax.set_ylabel('Y (px)')

plt.suptitle(f'Spatial Match Classification - {SAMPLE_ID}', fontsize=14)
plt.tight_layout()
plt.savefig(COMPARISON_DIR / 'spatial_match_map.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Summary

Key findings and recommendations.

In [ ]:
# Compile all results into a summary table
print("=" * 70)
print(f"  COMET vs STOmics Cellbin Comparison Summary — {SAMPLE_ID}")
print("=" * 70)

print(f"\n{'SEGMENTATION OVERVIEW':}")
print(f"  COMET cells:      {summary['n_comet']:>10,} (Visiopharm, warped via VALIS)")
print(f"  STOmics cells:    {summary['n_stomics']:>10,} (DAPI-derived cellbin)")
print(f"  COMET fg coverage: {overlap_metrics['comet_coverage_pct']:>9.1f}%")
print(f"  STOmics fg coverage: {overlap_metrics['stomics_coverage_pct']:>7.1f}%")

print(f"\n{'CELL MATCHING (max_distance={MAX_MATCH_DISTANCE}px)':}")
print(f"  1:1 mutual matches: {summary['n_1to1']:>8,}  ({summary['n_1to1']/summary['n_comet']*100:.1f}% of COMET)")
print(f"  COMET merged:       {summary['n_comet_merged']:>8,}  (multiple COMET → 1 STOmics)")
print(f"  COMET unmatched:    {summary['n_comet_only']:>8,}")
print(f"  STOmics fragmented: {summary['n_stomics_fragmented']:>8,}  (1 COMET → multiple STOmics)")
print(f"  STOmics unmatched:  {summary['n_stomics_only']:>8,}")
if len(matches_1to1) > 0:
    print(f"  Match distance:     median={matches_1to1['distance'].median():.1f}px, mean={matches_1to1['distance'].mean():.1f}px")

print(f"\n{'PIXEL OVERLAP':}")
print(f"  Foreground Jaccard: {overlap_metrics['foreground_jaccard']:.4f}")
print(f"  Pixel agreement:    {overlap_metrics['pixel_agreement']:.4f}")

if 'median_pearson_r' in expr_result['summary']:
    print(f"\n{'EXPRESSION CONCORDANCE':}")
    es = expr_result['summary']
    print(f"  Common genes:         {es['n_common_genes']:>8,}")
    print(f"  Pairs compared:       {es['n_pairs_compared']:>8,}")
    print(f"  Median per-cell r:    {es['median_pearson_r']:>8.3f}")
    print(f"  Cells with r > 0.5:   {es['pct_r_above_0.5']:>7.1f}%")
    print(f"  Median COMET counts:  {es['median_comet_counts']:>8.0f}")
    print(f"  Median STOmics counts:{es['median_stomics_counts']:>7.0f}")

print(f"\n{'OUTPUT FILES':}")
print(f"  {COMPARISON_DIR}/")
for f in sorted(COMPARISON_DIR.glob('*')):
    print(f"    {f.name} ({f.stat().st_size/1e6:.1f} MB)")

print("\n" + "=" * 70)
print("  Analysis complete. Review zoomed ROI overlays and spatial match maps")
print("  to assess alignment quality. Load QuPath GeoJSONs for detailed")
print("  visual inspection of cell boundary agreement.")
print("=" * 70)